# 06 — Causalidade de Granger: desastres naturais e inadimplência

Este notebook investiga, para o **período completo disponível**, se variações passadas na ocorrência nacional de desastres naturais acrescentam informação para prever variações futuras na taxa nacional de inadimplência de pessoas físicas.

> **Período analisado:** fevereiro de 2013 a dezembro de 2024. Janeiro de 2013 é perdido na construção da primeira diferença.

A investigação é organizada em três especificações complementares, apresentadas separadamente para que o papel do tratamento da sazonalidade seja avaliado com clareza.

Para a direção **desastres → inadimplência**:

- **H₀:** os coeficientes das defasagens de desastres são conjuntamente iguais a zero;
- **H₁:** pelo menos uma defasagem de desastres acrescenta informação preditiva para a inadimplência.

Também é testada a direção reversa, **inadimplência → desastres**, como diagnóstico. Causalidade de Granger significa precedência temporal e conteúdo preditivo, não causalidade estrutural.

## 1. Bibliotecas, configuração e caminhos

O notebook grava tabelas em `outputs/tables/` e figuras em `outputs/figures/`. As bases processadas são apenas lidas.

In [1]:
from pathlib import Path
from importlib.util import find_spec
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from IPython.display import display, Markdown
from scipy import stats
from statsmodels.api import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox, het_breuschpagan
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.graphics.tsaplots import plot_acf

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="The test statistic is outside")

SEED = 2026
np.random.seed(SEED)

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "notebooks" else CWD
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"
TABLES = OUTPUTS / "tables"
FIGURES = OUTPUTS / "figures"
REPORTS = OUTPUTS / "reports"
for pasta in (OUTPUTS, TABLES, FIGURES, REPORTS):
    pasta.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 180,
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
})

CORES = {"inad": "#1f4e79", "desastres": "#b3541e", "aic": "#6a51a3", "bic": "#238b45"}
print(f"Raiz do projeto: {ROOT}")
print(f"Statsmodels: {__import__('statsmodels').__version__}")

Raiz do projeto: /workspace/scratch/94bf83bea5df/repo
Statsmodels: 0.15.0


## 2. Bases e auditoria

São utilizadas duas bases produzidas pelo notebook 05:

1. **primeiras diferenças**, com
   
   $$\Delta I_t \quad\text{e}\quad \Delta\log(1+D_t);$$

2. **primeiras diferenças seguidas de diferença sazonal**, com
   
   $$\Delta_{12}\Delta I_t \quad\text{e}\quad \Delta_{12}\Delta\log(1+D_t).$$

A segunda base começa em fevereiro de 2014, pois perde mais 12 meses. A auditoria exige frequência mensal regular, ausência de duplicatas, valores ausentes e valores não finitos.

In [2]:
def ler_base(parquet, csv):
    if parquet.exists() and (find_spec("pyarrow") or find_spec("fastparquet")):
        return pd.read_parquet(parquet), parquet
    if csv.exists():
        return pd.read_csv(csv, sep=None, engine="python"), csv
    raise FileNotFoundError(f"Base não encontrada: {parquet.name} ou {csv.name}")

base, origem = ler_base(
    PROCESSED / "series_estacionarias_granger_2013_2024.parquet",
    PROCESSED / "series_estacionarias_granger_2013_2024.csv",
)
base_saz, origem_saz = ler_base(
    PROCESSED / "series_estacionarias_granger_sazonal_2013_2024.parquet",
    PROCESSED / "series_estacionarias_granger_sazonal_2013_2024.csv",
)

base = base[["data", "inad_diff1", "desastres_log1p_diff1"]].copy()
base_saz = base_saz[["data", "inad_diff1_diff12", "desastres_log1p_diff1_diff12"]].rename(columns={
    "inad_diff1_diff12": "inad_diff1",
    "desastres_log1p_diff1_diff12": "desastres_log1p_diff1",
})

for dados in (base, base_saz):
    dados["data"] = pd.to_datetime(dados["data"], errors="raise")
    dados.sort_values("data", inplace=True)
    dados.reset_index(drop=True, inplace=True)

def auditar(dados, nome, arquivo):
    grade = pd.date_range(dados["data"].min(), dados["data"].max(), freq="MS")
    linha = {
        "Base": nome,
        "Arquivo": arquivo.name,
        "Data inicial": dados["data"].min().date(),
        "Data final": dados["data"].max().date(),
        "Observações": len(dados),
        "Duplicatas": int(dados["data"].duplicated().sum()),
        "Meses ausentes": len(grade.difference(dados["data"])),
        "Ausentes": int(dados.isna().sum().sum()),
        "Não finitos": int((~np.isfinite(dados.iloc[:, 1:])).sum().sum()),
    }
    if any(linha[c] for c in ["Duplicatas", "Meses ausentes", "Ausentes", "Não finitos"]):
        raise ValueError(f"A base {nome} não passou pela auditoria.")
    return linha

auditoria = pd.DataFrame([
    auditar(base, "Primeiras diferenças", origem),
    auditar(base_saz, "Diferença sazonal das primeiras diferenças", origem_saz),
])
display(auditoria)
auditoria.to_csv(TABLES / "06_auditoria_bases.csv", index=False, encoding="utf-8-sig")

,Base,Arquivo,Data inicial,Data final,Observações,Duplicatas,Meses ausentes,Ausentes,Não finitos
0,Primeiras diferenças,series_estacionarias_granger_2013_2024.parquet,2013-02-01,2024-12-01,143,0,0,0,0
1,Diferença sazonal das primeiras diferenças,series_estacionarias_granger_sazonal_2013_2024...,2014-02-01,2024-12-01,131,0,0,0,0


## 3. Verificação de estacionariedade

O ADF testa a presença de raiz unitária, enquanto o KPSS testa estacionariedade. A leitura mais favorável ocorre quando o ADF rejeita sua hipótese nula e o KPSS não rejeita a própria hipótese nula.

Essa verificação confirma se as transformações empregadas no período analisado são adequadas ao teste de Granger.

In [3]:
def testar_estacionariedade(serie, base_nome, variavel):
    x = pd.Series(serie).dropna().astype(float)
    adf = adfuller(x, regression="c", autolag="AIC")
    kps = kpss(x, regression="c", nlags="auto")
    if adf[1] < .05 and kps[1] >= .05:
        conclusao = "Evidência convergente de estacionariedade"
    elif adf[1] >= .05 and kps[1] < .05:
        conclusao = "Evidência convergente de não estacionariedade"
    else:
        conclusao = "Resultado misto"
    return {
        "Base": base_nome, "Variável": variavel, "n": len(x),
        "ADF estatística": adf[0], "ADF p-valor": adf[1],
        "KPSS estatística": kps[0], "KPSS p-valor": kps[1],
        "Conclusão": conclusao,
    }

estacionariedade = pd.DataFrame([
    testar_estacionariedade(dados[coluna], nome, coluna)
    for nome, dados in [("Primeiras diferenças", base), ("Diferença sazonal", base_saz)]
    for coluna in ["inad_diff1", "desastres_log1p_diff1"]
])
display(estacionariedade.style.format({
    "ADF estatística": "{:.3f}", "ADF p-valor": "{:.4f}",
    "KPSS estatística": "{:.3f}", "KPSS p-valor": "{:.4f}",
}))
estacionariedade.to_csv(TABLES / "06_estacionariedade_periodo_total.csv", index=False, encoding="utf-8-sig")

,Base,Variável,n,ADF estatística,ADF p-valor,KPSS estatística,KPSS p-valor,Conclusão
0,Primeiras diferenças,inad_diff1,143,-3.215,0.0191,0.161,0.1000,Evidência convergente de estacionariedade
1,Primeiras diferenças,desastres_log1p_diff1,143,-6.965,0.0000,0.056,0.1000,Evidência convergente de estacionariedade
2,Diferença sazonal,inad_diff1,131,-3.860,0.0023,0.087,0.1000,Evidência convergente de estacionariedade
3,Diferença sazonal,desastres_log1p_diff1,131,-5.490,0.0000,0.329,0.1000,Evidência convergente de estacionariedade


## 4. Funções de estimação

Cada especificação é estimada como um sistema bivariado. Quando há dummies mensais, janeiro é a categoria de referência. A comparação restrita–irrestrita por OLS mantém os mesmos controles em ambas as equações.

Para cada especificação:

- AIC e BIC selecionam ordens separadamente;
- a **ordem BIC é adotada como principal**, por ser mais parcimoniosa;
- a ordem AIC é mantida como sensibilidade;
- p-valores exploratórios de todas as ordens recebem ajuste Benjamini–Hochberg;
- o teste da ordem selecionada também é recalculado com covariância HC3.

In [4]:
VARIAVEIS = ["inad_diff1", "desastres_log1p_diff1"]
ROTULOS = {"inad_diff1": "Inadimplência", "desastres_log1p_diff1": "Desastres"}
DIRECOES = [
    ("desastres → inadimplência", "desastres_log1p_diff1", "inad_diff1"),
    ("inadimplência → desastres", "inad_diff1", "desastres_log1p_diff1"),
]

def maximo_seguro(n, dummies, desejado=12):
    deterministicas = 12 if dummies else 1
    validos = []
    for p in range(1, desejado + 1):
        n_eff = n - p
        q = deterministicas + 2 * p
        if n_eff - q >= 30 and n_eff >= 3 * q:
            validos.append(p)
    return min(max(validos) if validos else 1, desejado)

def construir_design(dados, p, dummies, inicio_comum=None):
    y = dados.set_index("data")[VARIAVEIS].astype(float)
    X = pd.DataFrame({"const": 1.0}, index=y.index)
    if dummies:
        meses = pd.Categorical(y.index.month, categories=range(1, 13))
        dm = pd.get_dummies(meses, prefix="mes", drop_first=True, dtype=float)
        dm.index = y.index
        X = pd.concat([X, dm], axis=1)
    for lag in range(1, p + 1):
        for variavel in VARIAVEIS:
            X[f"{variavel}_lag{lag}"] = y[variavel].shift(lag)
    inicio = p if inicio_comum is None else inicio_comum
    X, y = X.iloc[inicio:].astype(float), y.iloc[inicio:].astype(float)
    if X.isna().any().any():
        raise ValueError("Há valores ausentes no desenho de regressão.")
    return y, X

def ajustar_sistema(dados, p, dummies, inicio_comum=None):
    y, X = construir_design(dados, p, dummies, inicio_comum)
    modelos = {alvo: OLS(y[alvo], X).fit() for alvo in VARIAVEIS}
    residuos = pd.DataFrame({alvo: modelos[alvo].resid for alvo in VARIAVEIS}, index=y.index)
    sigma = residuos.to_numpy().T @ residuos.to_numpy() / len(residuos)
    sinal, logdet = np.linalg.slogdet(sigma)
    if sinal <= 0:
        raise np.linalg.LinAlgError("Matriz residual não positiva definida.")
    return {"Y": y, "X": X, "modelos": modelos, "residuos": residuos, "logdet": logdet}

def selecionar_ordem(dados, dummies, max_lag):
    linhas = []
    k = len(VARIAVEIS)
    for p in range(1, max_lag + 1):
        ajuste = ajustar_sistema(dados, p, dummies, inicio_comum=max_lag)
        nobs, q, ld = len(ajuste["Y"]), ajuste["X"].shape[1], ajuste["logdet"]
        livres = k * q
        linhas.append({
            "Defasagem": p, "n comum": nobs, "Parâmetros por equação": q,
            "GL residuais": nobs - q,
            "AIC": ld + 2 * livres / nobs,
            "BIC": ld + np.log(nobs) * livres / nobs,
        })
    tabela = pd.DataFrame(linhas)
    escolhas = {c: int(tabela.loc[tabela[c].idxmin(), "Defasagem"]) for c in ["AIC", "BIC"]}
    tabela["Selecionada pelo AIC"] = tabela["Defasagem"].eq(escolhas["AIC"])
    tabela["Selecionada pelo BIC"] = tabela["Defasagem"].eq(escolhas["BIC"])
    return tabela, escolhas

def teste_granger(dados, p, causa, alvo, dummies):
    y, X = construir_design(dados, p, dummies)
    completo = OLS(y[alvo], X).fit()
    colunas = [f"{causa}_lag{lag}" for lag in range(1, p + 1)]
    restrito = OLS(y[alvo], X.drop(columns=colunas)).fit()
    f_stat, p_valor, gl_num = completo.compare_f_test(restrito)
    R = np.zeros((p, len(completo.params)))
    for i, coluna in enumerate(colunas):
        R[i, completo.model.exog_names.index(coluna)] = 1
    hc3 = completo.get_robustcov_results(cov_type="HC3").wald_test(R, use_f=True, scalar=True)
    return {
        "Estatística F": float(f_stat), "GL numerador": int(round(gl_num)),
        "GL denominador": int(round(completo.df_resid)), "p-valor": float(p_valor),
        "F HC3": float(hc3.statistic), "p-valor HC3": float(hc3.pvalue),
        "n efetivo": int(completo.nobs),
    }

def testes_por_lag(dados, dummies, max_lag):
    linhas = []
    for direcao, causa, alvo in DIRECOES:
        for p in range(1, max_lag + 1):
            linhas.append({"Direção": direcao, "Defasagem": p, **teste_granger(dados, p, causa, alvo, dummies)})
    tabela = pd.DataFrame(linhas)
    tabela["p-valor BH"] = tabela.groupby("Direção", sort=False)["p-valor"].transform(
        lambda x: multipletests(x, method="fdr_bh")[1]
    )
    tabela["Significativo BH 5%"] = tabela["p-valor BH"].lt(.05)
    return tabela

def estabilidade(modelos, p):
    k = len(VARIAVEIS)
    matrizes = []
    for lag in range(1, p + 1):
        A = np.zeros((k, k))
        for i, alvo in enumerate(VARIAVEIS):
            for j, origem in enumerate(VARIAVEIS):
                A[i, j] = modelos[alvo].params[f"{origem}_lag{lag}"]
        matrizes.append(A)
    companheira = matrizes[0] if p == 1 else np.vstack([
        np.hstack(matrizes),
        np.hstack([np.eye(k * (p - 1)), np.zeros((k * (p - 1), k))]),
    ])
    modulos = np.abs(np.linalg.eigvals(companheira))
    return float(modulos.max()), bool((modulos < 1).all())

def diagnosticar(dados, p, dummies):
    ajuste = ajustar_sistema(dados, p, dummies)
    raiz_max, estavel = estabilidade(ajuste["modelos"], p)
    linhas = []
    for alvo in VARIAVEIS:
        modelo = ajuste["modelos"][alvo]
        resid = modelo.resid
        lag_lb = min(max(12, p + 4), max(p + 1, len(resid) // 5))
        lb = acorr_ljungbox(resid, lags=[lag_lb], model_df=p, return_df=True).iloc[0]
        valores_acf = acf(resid, nlags=12, fft=False)
        limite = 1.96 / np.sqrt(len(resid))
        jb = stats.jarque_bera(resid)
        bp = het_breuschpagan(resid, modelo.model.exog)
        cooks = modelo.get_influence().cooks_distance[0]
        limiar_cook = 4 / len(resid)
        influentes = np.flatnonzero(cooks > limiar_cook)
        linhas.append({
            "Equação": ROTULOS[alvo], "Defasagem adotada": p,
            "Maior módulo da raiz": raiz_max, "Estável": estavel,
            "Ljung–Box p-valor": float(lb["lb_pvalue"]),
            "Autocorrelação residual": bool(lb["lb_pvalue"] < .05),
            "ACF(12) residual": float(valores_acf[12]), "Limite ACF 95%": limite,
            "Pico significativo no lag 12": bool(abs(valores_acf[12]) > limite),
            "Jarque–Bera p-valor": float(jb.pvalue),
            "Breusch–Pagan p-valor": float(bp[1]),
            "N observações influentes": int(len(influentes)),
        })
    return pd.DataFrame(linhas), ajuste

def executar_especificacao(chave, titulo, dados, dummies):
    max_lag = maximo_seguro(len(dados), dummies)
    criterios, escolhas = selecionar_ordem(dados, dummies, max_lag)
    testes = testes_por_lag(dados, dummies, max_lag)
    selecionados = []
    for criterio in ["AIC", "BIC"]:
        p = escolhas[criterio]
        parte = testes.loc[testes["Defasagem"].eq(p)].copy()
        parte.insert(0, "Critério", criterio)
        parte.insert(1, "Ordem selecionada", p)
        parte["Ordem adotada"] = criterio.eq("BIC") if hasattr(criterio, "eq") else criterio == "BIC"
        selecionados.append(parte)
    selecionados = pd.concat(selecionados, ignore_index=True)
    diag, ajuste = diagnosticar(dados, escolhas["BIC"], dummies)
    criterios.to_csv(TABLES / f"06_{chave}_criterios.csv", index=False, encoding="utf-8-sig")
    testes.to_csv(TABLES / f"06_{chave}_resultados_por_lag.csv", index=False, encoding="utf-8-sig")
    selecionados.to_csv(TABLES / f"06_{chave}_resultados_selecionados.csv", index=False, encoding="utf-8-sig")
    diag.to_csv(TABLES / f"06_{chave}_diagnosticos.csv", index=False, encoding="utf-8-sig")
    return {
        "chave": chave, "titulo": titulo, "dados": dados, "dummies": dummies,
        "max_lag": max_lag, "criterios": criterios, "escolhas": escolhas,
        "testes": testes, "selecionados": selecionados, "diagnosticos": diag, "ajuste": ajuste,
    }

def mostrar_criterios(resultado):
    escolhas, tab = resultado["escolhas"], resultado["criterios"]
    display(Markdown(
        f"> **Defasagem adotada: {escolhas['BIC']} mês(es), selecionada pelo BIC.**  \n"
        f"> O AIC selecionou {escolhas['AIC']} mês(es) e é apresentado como sensibilidade."
    ))
    display(tab.style.format({"AIC": "{:.4f}", "BIC": "{:.4f}"}).apply(
        lambda r: ["background-color:#d9ead3;font-weight:bold" if r["Selecionada pelo BIC"] else "" for _ in r], axis=1
    ))
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(tab["Defasagem"], tab["AIC"], marker="o", color=CORES["aic"], label=f"AIC — mínimo em p={escolhas['AIC']}")
    ax.plot(tab["Defasagem"], tab["BIC"], marker="o", color=CORES["bic"], label=f"BIC — mínimo em p={escolhas['BIC']}")
    ax.axvline(escolhas["BIC"], color=CORES["bic"], linestyle="--", alpha=.7)
    ax.set(title=f"Critérios de informação — {resultado['titulo']}", xlabel="Defasagem mensal", ylabel="Valor do critério")
    ax.set_xticks(tab["Defasagem"])
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURES / f"06_{resultado['chave']}_criterios.png", bbox_inches="tight")
    plt.show()

def mostrar_testes(resultado):
    tabela = resultado["selecionados"].copy()
    display(tabela[[
        "Critério", "Ordem selecionada", "Ordem adotada", "Direção", "Estatística F",
        "GL numerador", "GL denominador", "p-valor", "p-valor HC3", "p-valor BH",
    ]].style.format({
        "Estatística F": "{:.3f}", "p-valor": "{:.4f}",
        "p-valor HC3": "{:.4f}", "p-valor BH": "{:.4f}",
    }).apply(lambda r: [
        "background-color:#d9ead3;font-weight:bold" if r["Ordem adotada"] else "" for _ in r
    ], axis=1))

def mostrar_diagnosticos(resultado):
    display(resultado["diagnosticos"].style.format({
        "Maior módulo da raiz": "{:.3f}", "Ljung–Box p-valor": "{:.4f}",
        "ACF(12) residual": "{:.3f}", "Limite ACF 95%": "{:.3f}",
        "Jarque–Bera p-valor": "{:.4f}", "Breusch–Pagan p-valor": "{:.4f}",
    }))
    ajuste = resultado["ajuste"]
    fig, axes = plt.subplots(3, 2, figsize=(14, 10))
    for j, alvo in enumerate(VARIAVEIS):
        modelo = ajuste["modelos"][alvo]
        resid = modelo.resid
        datas = ajuste["Y"].index
        axes[0, j].plot(datas, resid, color=CORES["inad"] if j == 0 else CORES["desastres"])
        axes[0, j].axhline(0, color="black", linewidth=.8)
        axes[0, j].set_title(f"Resíduos no tempo — {ROTULOS[alvo]}")
        axes[1, j].scatter(modelo.fittedvalues, resid, alpha=.7, s=25)
        axes[1, j].axhline(0, color="black", linewidth=.8)
        axes[1, j].set_title(f"Resíduos × ajustados — {ROTULOS[alvo]}")
        plot_acf(resid, lags=min(24, len(resid) // 3), zero=False, ax=axes[2, j], alpha=.05)
        axes[2, j].axvline(12, color="crimson", linestyle="--", label="lag 12")
        axes[2, j].set_title(f"ACF dos resíduos — {ROTULOS[alvo]}")
        axes[2, j].legend()
    fig.suptitle(f"Diagnósticos da ordem BIC — {resultado['titulo']}", y=1.01, fontsize=15, fontweight="bold")
    fig.tight_layout()
    fig.savefig(FIGURES / f"06_{resultado['chave']}_diagnosticos.png", bbox_inches="tight")
    plt.show()

def interpretar(resultado, observacao_extra=""):
    tab = resultado["selecionados"]
    textos = []
    for criterio in ["AIC", "BIC"]:
        principal = tab.loc[(tab["Critério"] == criterio) & (tab["Direção"] == "desastres → inadimplência")].iloc[0]
        reversa = tab.loc[(tab["Critério"] == criterio) & (tab["Direção"] == "inadimplência → desastres")].iloc[0]
        decisao = "rejeita" if principal["p-valor"] < .05 else "não rejeita"
        decisao_reversa = "rejeita" if reversa["p-valor"] < .05 else "não rejeita"
        textos.append(
            f"- **{criterio}, p={int(principal['Ordem selecionada'])}:** na direção principal, {decisao} H₀ a 5% "
            f"(p = {principal['p-valor']:.4f}; p-HC3 = {principal['p-valor HC3']:.4f}; p-BH = {principal['p-valor BH']:.4f}). "
            f"Na direção reversa, {decisao_reversa} H₀ (p = {reversa['p-valor']:.4f}; "
            f"p-HC3 = {reversa['p-valor HC3']:.4f}; p-BH = {reversa['p-valor BH']:.4f})."
        )
    diag_inad = resultado["diagnosticos"].loc[resultado["diagnosticos"]["Equação"] == "Inadimplência"].iloc[0]
    diag_des = resultado["diagnosticos"].loc[resultado["diagnosticos"]["Equação"] == "Desastres"].iloc[0]
    sazonal = "há" if diag_inad["Pico significativo no lag 12"] else "não há"
    autocorr = "há" if diag_inad["Autocorrelação residual"] else "não há"
    normalidade = "é rejeitada" if diag_inad["Jarque–Bera p-valor"] < .05 else "não é rejeitada"
    hetero = "é detectada" if diag_inad["Breusch–Pagan p-valor"] < .05 else "não é detectada"
    conclusao = (
        f"Na ordem adotada pelo BIC, o sistema é {'estável' if diag_inad['Estável'] else 'instável'}. "
        f"Na equação da inadimplência, {autocorr} autocorrelação residual pelo Ljung–Box e {sazonal} pico significativo "
        f"na ACF(12). A normalidade {normalidade} pelo Jarque–Bera e a heterocedasticidade {hetero} pelo Breusch–Pagan. "
        f"Foram sinalizadas {int(diag_inad['N observações influentes'])} observações potencialmente influentes. "
        f"Na equação dos desastres, o Ljung–Box {'detecta' if diag_des['Autocorrelação residual'] else 'não detecta'} "
        f"autocorrelação residual."
    )
    display(Markdown("### Interpretação desta especificação\n\n" + "\n".join(textos) + "\n\n" + conclusao + observacao_extra))

---

## 5. Especificação 1 — Principal: primeiras diferenças com 11 dummies mensais

Esta é a especificação principal. As dummies controlam diferenças médias entre janeiro, fevereiro, março etc., evitando que um calendário compartilhado seja confundido com precedência temporal.

**Base:** $\Delta I_t$ e $\Delta\log(1+D_t)$, fevereiro/2013–dezembro/2024.  
**Controles determinísticos:** constante e 11 dummies mensais.  
**Ordem principal:** selecionada pelo BIC; AIC é sensibilidade.

In [5]:
spec1 = executar_especificacao(
    "spec1_dummies", "Primeiras diferenças com 11 dummies mensais", base, dummies=True
)
mostrar_criterios(spec1)

> **Defasagem adotada: 1 mês(es), selecionada pelo BIC.**  
> O AIC selecionou 2 mês(es) e é apresentado como sensibilidade.

,Defasagem,n comum,Parâmetros por equação,GL residuais,AIC,BIC,Selecionada pelo AIC,Selecionada pelo BIC
0,1,131,14,117,-6.1564,-5.5419,False,True
1,2,131,16,115,-6.2340,-5.5317,True,False
2,3,131,18,113,-6.2007,-5.4106,False,False
3,4,131,20,111,-6.1838,-5.3059,False,False
4,5,131,22,109,-6.2048,-5.2391,False,False
5,6,131,24,107,-6.1763,-5.1228,False,False
6,7,131,26,105,-6.1758,-5.0345,False,False
7,8,131,28,103,-6.1478,-4.9187,False,False
8,9,131,30,101,-6.1455,-4.8286,False,False
9,10,131,32,99,-6.1439,-4.7392,False,False


<Figure>

In [6]:
mostrar_testes(spec1)

,Critério,Ordem selecionada,Ordem adotada,Direção,Estatística F,GL numerador,GL denominador,p-valor,p-valor HC3,p-valor BH
0,AIC,2,False,desastres → inadimplência,0.858,2,125,0.4267,0.4752,0.9397
1,AIC,2,False,inadimplência → desastres,0.471,2,125,0.6253,0.6362,0.8338
2,BIC,1,True,desastres → inadimplência,1.119,1,128,0.2921,0.3003,0.9397
3,BIC,1,True,inadimplência → desastres,0.673,1,128,0.4137,0.4793,0.6821


In [7]:
mostrar_diagnosticos(spec1)
interpretar(spec1)

,Equação,Defasagem adotada,Maior módulo da raiz,Estável,Ljung–Box p-valor,Autocorrelação residual,ACF(12) residual,Limite ACF 95%,Pico significativo no lag 12,Jarque–Bera p-valor,Breusch–Pagan p-valor,N observações influentes
0,Inadimplência,1,0.451,True,0.1872,False,-0.055,0.164,False,0.0000,0.5185,7
1,Desastres,1,0.451,True,0.0329,True,0.063,0.164,False,0.0938,0.6461,10


<Figure>

### Interpretação desta especificação

- **AIC, p=2:** na direção principal, não rejeita H₀ a 5% (p = 0.4267; p-HC3 = 0.4752; p-BH = 0.9397). Na direção reversa, não rejeita H₀ (p = 0.6253; p-HC3 = 0.6362; p-BH = 0.8338).
- **BIC, p=1:** na direção principal, não rejeita H₀ a 5% (p = 0.2921; p-HC3 = 0.3003; p-BH = 0.9397). Na direção reversa, não rejeita H₀ (p = 0.4137; p-HC3 = 0.4793; p-BH = 0.6821).

Na ordem adotada pelo BIC, o sistema é estável. Na equação da inadimplência, não há autocorrelação residual pelo Ljung–Box e não há pico significativo na ACF(12). A normalidade é rejeitada pelo Jarque–Bera e a heterocedasticidade não é detectada pelo Breusch–Pagan. Foram sinalizadas 7 observações potencialmente influentes. Na equação dos desastres, o Ljung–Box detecta autocorrelação residual.

---

## 6. Especificação 2 — Comparação: primeiras diferenças sem controle sazonal

Esta especificação usa as mesmas séries da principal, mas remove as dummies mensais. Ela serve para mostrar o que acontece quando a sazonalidade determinística é ignorada.

**Base:** $\Delta I_t$ e $\Delta\log(1+D_t)$, fevereiro/2013–dezembro/2024.  
**Controles determinísticos:** apenas constante.  
**Uso:** comparação; não substitui automaticamente o modelo principal.

In [8]:
spec2 = executar_especificacao(
    "spec2_sem_controle", "Primeiras diferenças sem controle sazonal", base, dummies=False
)
mostrar_criterios(spec2)

> **Defasagem adotada: 1 mês(es), selecionada pelo BIC.**  
> O AIC selecionou 7 mês(es) e é apresentado como sensibilidade.

,Defasagem,n comum,Parâmetros por equação,GL residuais,AIC,BIC,Selecionada pelo AIC,Selecionada pelo BIC
0,1,131,3,128,-5.6076,-5.4760,False,True
1,2,131,5,126,-5.6670,-5.4475,False,False
2,3,131,7,124,-5.7311,-5.4238,False,False
3,4,131,9,122,-5.7896,-5.3945,False,False
4,5,131,11,120,-5.8389,-5.3561,False,False
5,6,131,13,118,-5.9158,-5.3452,False,False
6,7,131,15,116,-5.9523,-5.2939,True,False
7,8,131,17,114,-5.9324,-5.1861,False,False
8,9,131,19,112,-5.9474,-5.1134,False,False
9,10,131,21,110,-5.9253,-5.0034,False,False


<Figure>

In [9]:
mostrar_testes(spec2)

,Critério,Ordem selecionada,Ordem adotada,Direção,Estatística F,GL numerador,GL denominador,p-valor,p-valor HC3,p-valor BH
0,AIC,7,False,desastres → inadimplência,0.831,7,121,0.5634,0.6464,0.7547
1,AIC,7,False,inadimplência → desastres,2.337,7,121,0.0284,0.0050,0.0487
2,BIC,1,True,desastres → inadimplência,0.857,1,139,0.3562,0.2990,0.7547
3,BIC,1,True,inadimplência → desastres,5.058,1,139,0.0261,0.0534,0.0487


In [10]:
mostrar_diagnosticos(spec2)
interpretar(
    spec2,
    " Como esta especificação ignora diferenças sistemáticas entre os meses, qualquer significância exclusiva daqui deve ser tratada como possível confusão sazonal."
)

,Equação,Defasagem adotada,Maior módulo da raiz,Estável,Ljung–Box p-valor,Autocorrelação residual,ACF(12) residual,Limite ACF 95%,Pico significativo no lag 12,Jarque–Bera p-valor,Breusch–Pagan p-valor,N observações influentes
0,Inadimplência,1,0.329,True,0.0000,True,0.391,0.164,True,0.0002,0.1979,7
1,Desastres,1,0.329,True,0.0088,True,0.189,0.164,True,0.8701,0.0794,7


<Figure>

### Interpretação desta especificação

- **AIC, p=7:** na direção principal, não rejeita H₀ a 5% (p = 0.5634; p-HC3 = 0.6464; p-BH = 0.7547). Na direção reversa, rejeita H₀ (p = 0.0284; p-HC3 = 0.0050; p-BH = 0.0487).
- **BIC, p=1:** na direção principal, não rejeita H₀ a 5% (p = 0.3562; p-HC3 = 0.2990; p-BH = 0.7547). Na direção reversa, rejeita H₀ (p = 0.0261; p-HC3 = 0.0534; p-BH = 0.0487).

Na ordem adotada pelo BIC, o sistema é estável. Na equação da inadimplência, há autocorrelação residual pelo Ljung–Box e há pico significativo na ACF(12). A normalidade é rejeitada pelo Jarque–Bera e a heterocedasticidade não é detectada pelo Breusch–Pagan. Foram sinalizadas 7 observações potencialmente influentes. Na equação dos desastres, o Ljung–Box detecta autocorrelação residual. Como esta especificação ignora diferenças sistemáticas entre os meses, qualquer significância exclusiva daqui deve ser tratada como possível confusão sazonal.

---

## 7. Especificação 3 — Robustez: $\Delta_{12}\Delta$ sem dummies mensais

Nesta especificação, a sazonalidade é tratada pela diferença sazonal aplicada às primeiras diferenças:

$$
\Delta_{12}\Delta I_t
\quad\text{e}\quad
\Delta_{12}\Delta\log(1+D_t).
$$

**Base:** fevereiro/2014–dezembro/2024.  
**Controles determinísticos:** apenas constante.  
**Custo:** perda adicional de 12 meses e possibilidade de sobrediferenciação.  
**Uso:** robustez; não substitui automaticamente a especificação com dummies.

In [11]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(base_saz["data"], base_saz["inad_diff1"], color=CORES["inad"])
axes[0].axhline(0, color="black", linewidth=.8)
axes[0].set(title="Diferença sazonal da variação mensal da inadimplência", ylabel="Pontos percentuais")
axes[1].plot(base_saz["data"], base_saz["desastres_log1p_diff1"], color=CORES["desastres"])
axes[1].axhline(0, color="black", linewidth=.8)
axes[1].set(title="Diferença sazonal da variação logarítmica dos desastres", ylabel="Variação logarítmica", xlabel="Mês")
fig.tight_layout()
fig.savefig(FIGURES / "06_spec3_series.png", bbox_inches="tight")
plt.show()

acf_entrada = []
for coluna in VARIAVEIS:
    x = base_saz[coluna].astype(float)
    valores = acf(x, nlags=12, fft=False)
    limite = 1.96 / np.sqrt(len(x))
    acf_entrada.append({
        "Série": ROTULOS[coluna], "ACF(12)": valores[12], "Limite 95%": limite,
        "Significativa": bool(abs(valores[12]) > limite),
    })
acf_entrada = pd.DataFrame(acf_entrada)
display(Markdown("### Diagnóstico da transformação sazonal"))
display(acf_entrada.style.format({"ACF(12)": "{:.3f}", "Limite 95%": "{:.3f}"}))
acf_entrada.to_csv(TABLES / "06_spec3_acf_series.csv", index=False, encoding="utf-8-sig")

<Figure>

### Diagnóstico da transformação sazonal

,Série,ACF(12),Limite 95%,Significativa
0,Inadimplência,-0.438,0.171,True
1,Desastres,-0.274,0.171,True


In [12]:
spec3 = executar_especificacao(
    "spec3_diferenca_sazonal", "Diferença sazonal das primeiras diferenças", base_saz, dummies=False
)
mostrar_criterios(spec3)

> **Defasagem adotada: 2 mês(es), selecionada pelo BIC.**  
> O AIC selecionou 12 mês(es) e é apresentado como sensibilidade.

,Defasagem,n comum,Parâmetros por equação,GL residuais,AIC,BIC,Selecionada pelo AIC,Selecionada pelo BIC
0,1,119,3,116,-5.0987,-4.9586,False,False
1,2,119,5,114,-5.1990,-4.9655,False,True
2,3,119,7,112,-5.1639,-4.8369,False,False
3,4,119,9,110,-5.1381,-4.7177,False,False
4,5,119,11,108,-5.1222,-4.6084,False,False
5,6,119,13,106,-5.0838,-4.4766,False,False
6,7,119,15,104,-5.0809,-4.3803,False,False
7,8,119,17,102,-5.0342,-4.2402,False,False
8,9,119,19,100,-5.0334,-4.1460,False,False
9,10,119,21,98,-5.0889,-4.1080,False,False


<Figure>

In [13]:
mostrar_testes(spec3)

,Critério,Ordem selecionada,Ordem adotada,Direção,Estatística F,GL numerador,GL denominador,p-valor,p-valor HC3,p-valor BH
0,AIC,12,False,desastres → inadimplência,1.280,12,94,0.2436,0.1984,0.5160
1,AIC,12,False,inadimplência → desastres,3.006,12,94,0.0013,0.0023,0.0161
2,BIC,2,True,desastres → inadimplência,1.129,2,124,0.3268,0.3608,0.5160
3,BIC,2,True,inadimplência → desastres,0.251,2,124,0.7786,0.7832,0.9202


In [14]:
mostrar_diagnosticos(spec3)
negativa_significativa = bool((acf_entrada["Significativa"] & acf_entrada["ACF(12)"].lt(0)).all())
extra = (
    " As duas séries transformadas apresentam ACF(12) negativa e significativa, padrão compatível com sobrediferenciação sazonal; portanto, esta especificação deve permanecer como robustez."
    if negativa_significativa else " A diferença sazonal não produz o mesmo padrão de ACF(12) nas duas séries."
)
interpretar(spec3, extra)

,Equação,Defasagem adotada,Maior módulo da raiz,Estável,Ljung–Box p-valor,Autocorrelação residual,ACF(12) residual,Limite ACF 95%,Pico significativo no lag 12,Jarque–Bera p-valor,Breusch–Pagan p-valor,N observações influentes
0,Inadimplência,2,0.651,True,0.0002,True,-0.379,0.173,True,0.0384,0.8458,7
1,Desastres,2,0.651,True,0.0055,True,-0.301,0.173,True,0.6497,0.5950,11


<Figure>

### Interpretação desta especificação

- **AIC, p=12:** na direção principal, não rejeita H₀ a 5% (p = 0.2436; p-HC3 = 0.1984; p-BH = 0.5160). Na direção reversa, rejeita H₀ (p = 0.0013; p-HC3 = 0.0023; p-BH = 0.0161).
- **BIC, p=2:** na direção principal, não rejeita H₀ a 5% (p = 0.3268; p-HC3 = 0.3608; p-BH = 0.5160). Na direção reversa, não rejeita H₀ (p = 0.7786; p-HC3 = 0.7832; p-BH = 0.9202).

Na ordem adotada pelo BIC, o sistema é estável. Na equação da inadimplência, há autocorrelação residual pelo Ljung–Box e há pico significativo na ACF(12). A normalidade é rejeitada pelo Jarque–Bera e a heterocedasticidade não é detectada pelo Breusch–Pagan. Foram sinalizadas 7 observações potencialmente influentes. Na equação dos desastres, o Ljung–Box detecta autocorrelação residual. As duas séries transformadas apresentam ACF(12) negativa e significativa, padrão compatível com sobrediferenciação sazonal; portanto, esta especificação deve permanecer como robustez.

---

## 8. Síntese final das três especificações

As especificações foram apresentadas separadamente para que seus pressupostos e resultados não sejam confundidos. A síntese abaixo compara apenas as conclusões substantivas, mantendo a especificação 1 como referência principal.

In [15]:
def resumo_especificacao(resultado):
    linha = resultado["selecionados"].loc[
        (resultado["selecionados"]["Critério"] == "BIC")
        & (resultado["selecionados"]["Direção"] == "desastres → inadimplência")
    ].iloc[0]
    return {
        "Especificação": resultado["titulo"],
        "Ordem BIC adotada": int(linha["Ordem selecionada"]),
        "p-valor": linha["p-valor"],
        "p-valor HC3": linha["p-valor HC3"],
        "p-valor BH": linha["p-valor BH"],
        "Conclusão a 5%": "Evidência no sentido de Granger" if linha["p-valor"] < .05 else "Sem evidência estatística",
    }

resumos = [resumo_especificacao(x) for x in [spec1, spec2, spec3]]
for resumo in resumos:
    display(Markdown(
        f"### {resumo['Especificação']}\n\n"
        f"- **Defasagem BIC adotada:** {resumo['Ordem BIC adotada']} mês(es).\n"
        f"- **p-valor clássico:** {resumo['p-valor']:.4f}.\n"
        f"- **p-valor HC3:** {resumo['p-valor HC3']:.4f}.\n"
        f"- **p-valor BH:** {resumo['p-valor BH']:.4f}.\n"
        f"- **Conclusão:** {resumo['Conclusão a 5%']}."
    ))

principal = resumos[0]
display(Markdown(
    "## Resposta à pergunta principal\n\n"
    "A especificação principal com 11 dummies mensais não encontrou evidência de que desastres naturais precedam "
    "a inadimplência no sentido de Granger. A comparação sem dummies e a robustez com diferença sazonal também "
    "não alteram essa conclusão para a direção principal. Isso representa **ausência de evidência estatística na "
    "amostra e nas especificações avaliadas**, e não prova de que desastres nunca tenham efeito sobre a inadimplência."
))

### Primeiras diferenças com 11 dummies mensais

- **Defasagem BIC adotada:** 1 mês(es).
- **p-valor clássico:** 0.2921.
- **p-valor HC3:** 0.3003.
- **p-valor BH:** 0.9397.
- **Conclusão:** Sem evidência estatística.

### Primeiras diferenças sem controle sazonal

- **Defasagem BIC adotada:** 1 mês(es).
- **p-valor clássico:** 0.3562.
- **p-valor HC3:** 0.2990.
- **p-valor BH:** 0.7547.
- **Conclusão:** Sem evidência estatística.

### Diferença sazonal das primeiras diferenças

- **Defasagem BIC adotada:** 2 mês(es).
- **p-valor clássico:** 0.3268.
- **p-valor HC3:** 0.3608.
- **p-valor BH:** 0.5160.
- **Conclusão:** Sem evidência estatística.

## Resposta à pergunta principal

A especificação principal com 11 dummies mensais não encontrou evidência de que desastres naturais precedam a inadimplência no sentido de Granger. A comparação sem dummies e a robustez com diferença sazonal também não alteram essa conclusão para a direção principal. Isso representa **ausência de evidência estatística na amostra e nas especificações avaliadas**, e não prova de que desastres nunca tenham efeito sobre a inadimplência.

## 9. Limitações e próximos passos

1. O agregado nacional pode ocultar efeitos concentrados em determinados estados, tipos de desastre ou níveis de severidade.
2. A contagem de ocorrências não mede diretamente pessoas afetadas, prejuízos ou duração do evento.
3. Granger avalia precedência temporal, não causalidade estrutural.
4. A especificação sem dummies pode confundir sazonalidade com ganho preditivo.
5. A diferença sazonal reduz a amostra e pode gerar sobrediferenciação.
6. Resultados exploratórios por defasagem não devem substituir a ordem selecionada previamente.

Uma extensão natural seria um painel mensal por UF, com medidas de intensidade do desastre e controles econômicos, sem alterar a conclusão deste notebook nacional.

## 10. Arquivos produzidos

Para cada especificação são gerados arquivos separados de critérios, resultados por lag, resultados selecionados e diagnósticos, todos em `outputs/tables/`. As figuras também recebem prefixos distintos:

- `06_spec1_dummies_*`;
- `06_spec2_sem_controle_*`;
- `06_spec3_diferenca_sazonal_*`.

O relatório completo é exportado para `outputs/reports/06_causalidade_granger.html`.